# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "PatchTST_OLCV"
REPO_DIR = "/content/ECE1508_GenAI"   # absolute path -- see note below

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    # git -C targets REPO_DIR explicitly rather than `cd X && ...`, so this is correct
    # regardless of the kernel's current working directory when the cell reruns.
    !git -C {REPO_DIR} pull

# Absolute path, not "ECE1508_GenAI": os.path.isdir("ECE1508_GenAI") above is checked
# relative to the CURRENT working directory -- on a second run of this cell (after the
# %cd below already moved the kernel into /content/ECE1508_GenAI), that relative check
# looks for /content/ECE1508_GenAI/ECE1508_GenAI, finds nothing, and silently clones a
# second, nested copy of the repo inside the first one instead of pulling it.
%cd {REPO_DIR}

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 89.21 KiB | 1.56 MiB/s, done.
From https://github.com/WoodyChang21/ECE1508_GenAI
   f198778..dc5c02f  PatchTST_OLCV -> origin/PatchTST_OLCV
Updating f198778..dc5c02f
Fast-forward
 ...train_patchtst_hf_channel_attention_False.ipynb | 590 +++++++++++++++++++--
 .../train_patchtst_hf_channel_attention_True.ipynb | 572 +++++++++++++++++++-
 2 files changed, 1104 insertions(+), 58 deletions(-)
/content/ECE1508_GenAI


In [3]:
# torch is preinstalled on Colab; transformers is needed for the HF PatchTSTModel-backed
# train_patchtst.py (src/models/patchtst_hf.py); mplfinance/pyyaml are for evaluate.py/configs.
!pip install -q transformers mplfinance pyyaml

In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [5]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: anyio-4.14.2, typeguard-4.5.2, langsmith-0.10.2
collected 17 items                                                             

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! KeyboardInterrupt !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
<frozen importlib._bootstrap>:1334: KeyboardInterrupt
(to show a full traceback on KeyboardInterrupt use --full-trace)
============================ no tests ran in 6.21s =============================


## Train PatchTST (HF PatchTSTModel-backed, real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Now trains `src/models/patchtst_hf.py` instead of the original hand-rolled `src/models/patchtst.py`
-- see `docs/experiments.md` for the comparison. `channel_attention=False` is the config default
(the main approach for now); add `--channel-attention` below to try the mixing variant instead.
Saves to `steven/outputs/patchtst_hf_checkpoint.pt` (separate from the original model's checkpoint).

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first
instead of the full config.

In [ ]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

## Evaluate all three checkpoints on the fixed test set

`evaluate.py` auto-detects which architecture a checkpoint holds (original hand-rolled vs.
HF PatchTSTModel-backed, see `detect_patchtst_arch`) and auto-derives separate output paths
from the checkpoint's filename (see `derive_output_suffix`), so running all three below in
order is safe -- none of them overwrite each other's results:

| `--patchtst-checkpoint` | writes to |
|---|---|
| `patchtst_checkpoint.pt` (original) | `metrics.json`, `sample_plots/` |
| `patchtst_false_checkpoint.pt` (HF, `channel_attention=False`) | `metrics_false.json`, `sample_plots_false/` |
| `patchtst_true_checkpoint.pt` (HF, `channel_attention=True`) | `metrics_true.json`, `sample_plots_true/` |

For the HF checkpoints specifically: that architecture only supports its trained fixed
`context_length` (no padding support), so test/plot windows are restricted to that fixed
length instead of the original's 2-10 day curriculum -- all those windows land in the
`long` bucket in the output, compare against that bucket, not `overall`.</cell id="58857849">


In [1]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_false_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt

python3: can't open file '/content/steven/src/evaluate.py': [Errno 2] No such file or directory


In [8]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_true_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt

00:46:58 metrics-out: steven/outputs/metrics_true.json, plots-dir: steven/outputs/sample_plots_true
00:46:58 device: cuda
00:46:58 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
00:46:58 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
00:46:58 sell-price shrink bound: p99.0 of |anchored log return| over train = 0.0190 (vs. model's own MAX_LOG_RETURN)
00:46:58 patchtst checkpoint architecture: hf
00:46:58 evaluating on 2397 fixed test windows
00:46:59 wrote metrics to steven/outputs/metrics_true.json
00:46:59 overall: {
  "n_windows": 2397,
  "patchtst_reparam_mae_rmse": [
    0.11513102799654007,
    0.48324185609817505
  ],
  "cvae_reparam_mae_rmse": [
    0.14826694130897522,
    0.4976522922515869
  ],
  "patchtst_ohlc_mae_rmse": [
    3.5796957492326,
    4.6973235095607
  ],
  "cvae_ohlc_mae_rmse": [
    3.1049627297287197,
    4.188423817430327
  ],
  "patchtst_volume_mae_rmse": [
    2865898.5,
    9654751.0
  ],
  "cvae_

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [7]:
!python steven/src/update_report.py

22:34:43 updated steven/v1.md: results-samples, hit-summary, spread-summary, backtest-patchtst, backtest-cvae, buy-hold-benchmark
22:34:43 not auto-updated -- reread and edit by hand if the story changed: the 'In plain terms' / 'A subtle but important point' interpretation paragraphs under Results, the 'pre-retrain checkpoints' caveats in Results and Long-only backtest results, and the 'Retrain both models' checkbox under Next steps.


## Pull results back down

Zips `steven/outputs/` (checkpoints, metrics.json, sample_plots) and downloads it -- or just `git add`/`commit`/`push` from here if you'd rather sync back through the repo.

In [ ]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

### Or: commit results straight back to the repo (recommended)

`files.download()` above requires Colab's own hosted web frontend and won't trigger a
download when connected via a different client (e.g. Cursor's Colab GPU extension) --
if nothing downloaded from the cell above, use this instead. Commits
`steven/outputs/` (checkpoints, `metrics.json`, `sample_plots/`) and `steven/v1.md`
from this Colab session directly to `origin/{BRANCH}`; pull locally afterward to sync.

In [9]:
# Fresh Colab VM has no git identity configured -- needed for `commit` to work at all.
# Only sets it for this local clone (no --global), harmless to commit/share.
!git -C {REPO_DIR} config user.email "woodychang891121@gmail.com"
!git -C {REPO_DIR} config user.name "WoodyChang21"

!git add steven/outputs steven/v1.md
!git status
!git commit -m "chore(model): update metrics.json/sample_plots/v1.md from this Colab run"

On branch PatchTST_OLCV
Your branch is up to date with 'origin/PatchTST_OLCV'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   steven/outputs/metrics.json
	modified:   steven/outputs/metrics_false.json
	modified:   steven/outputs/metrics_true.json
	new file:   steven/outputs/patchtst_false_headindep_checkpoint.pt
	new file:   steven/outputs/patchtst_false_priceonly_checkpoint.pt
	new file:   steven/outputs/patchtst_true_headindep_checkpoint.pt
	new file:   steven/outputs/patchtst_true_priceonly_checkpoint.pt
	new file:   steven/outputs/sample_plots/sample0_start25519_ctx21.png
	deleted:    steven/outputs/sample_plots/sample0_start26021_ctx70.png
	new file:   steven/outputs/sample_plots/sample1_start26087_ctx56.png
	deleted:    steven/outputs/sample_plots/sample1_start26901_ctx70.png
	deleted:    steven/outputs/sample_plots/sample2_start26298_ctx70.png
	new file:   steven/outputs/sample_plots/sample2_start26324_ctx35.png
	deleted:    steven/out

In [10]:
import getpass

# Fresh Colab VM has no stored GitHub credentials, so a plain `git push` over HTTPS
# can't authenticate. Prompting interactively (getpass masks it, and it's never written
# into this notebook's saved source/outputs) instead of hardcoding a token in a cell --
# a hardcoded token would get committed into git history the moment this notebook is
# pushed, which is a real credential leak. Needs a GitHub Personal Access Token with
# `repo` scope: https://github.com/settings/tokens
token = getpass.getpass("GitHub Personal Access Token: ")
push_url = f"https://{token}@github.com/WoodyChang21/ECE1508_GenAI.git"
!git -C {REPO_DIR} push {push_url} {BRANCH}
del token, push_url  # don't leave it sitting in a notebook-visible variable longer than needed

Enumerating objects: 43, done.
Counting objects: 100% (43/43), done.
Delta compression using up to 12 threads
Compressing objects: 100% (32/32), done.
Writing objects: 100% (32/32), 5.24 MiB | 8.19 MiB/s, done.
Total 32 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/WoodyChang21/ECE1508_GenAI.git
   dc5c02f..5fb9a81  PatchTST_OLCV -> PatchTST_OLCV
